# Line

An interval of the real line: a wall, an electrode or an absorber in a 1D domain.

`Line(midpoint, side_lengths)` covers `midpoint ± side_lengths / 2`. Both are one-element
sequences, as for `Rectangle` and `Cuboid` in two and three dimensions. Points are `(1, n)`
arrays, component-major like every query in simgeometry.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from simgeometry import Grid, Line

wall = Line(midpoint=(0.6,), side_lengths=(0.25,))
print("from", float(wall.lower_bound[0]), "to", float(wall.upper_bound[0]))
print("x = 0.6 inside:", wall.contains((0.6,)))
print("points inside:", wall.contains_points(np.array([[0.1, 0.5, 0.7, 0.9]])))
assert wall.contains((0.6,)) and not wall.contains((0.2,))

## Node mask and level set

`node_mask(grid)` marks the grid nodes inside the line. Its level set is the exact signed
distance to the interval: negative inside, zero at both ends, growing with slope one outside.

In [ ]:
grid = Grid(1, (0.0,), (1.0,), (21,))
mask = wall.node_mask(grid)
nodes = grid.node_points()[0]
print("nodes inside:", nodes[mask])

x = np.linspace(0.0, 1.0, 401)
distance = wall.signed_distance(x[None, :])
assert np.isclose(wall.signed_distance((0.2,)), 0.6 - 0.125 - 0.2)

fig, ax = plt.subplots(figsize=(6, 3), layout="constrained")
ax.axvspan(wall.lower_bound[0], wall.upper_bound[0], color="tab:orange", alpha=0.25)
ax.plot(x, distance, label="signed distance")
ax.axhline(0.0, color="0.5", lw=0.8)
ax.plot(
    nodes[~mask], np.zeros((~mask).sum()), "o", mfc="none", color="0.5", label="nodes"
)
ax.plot(
    nodes[mask], np.zeros(mask.sum()), "o", color="tab:orange", label="nodes inside"
)
ax.set(xlabel="x", ylabel="signed distance", title="A Line and its level set")
ax.legend(loc="upper center")
plt.show()

## Inverted lines

`inverted=True` covers everything except the interval, for example the walls on both sides of
a gap. The node mask is the complement and the level set changes sign.

In [ ]:
gap = Line(midpoint=(0.6,), side_lengths=(0.25,), inverted=True)
assert np.array_equal(gap.node_mask(grid), ~mask)
assert np.isclose(gap.signed_distance((0.2,)), -wall.signed_distance((0.2,)))
print("walls at x = 0.2:", gap.contains((0.2,)))